# Practice 4-1. 부모-자식 분할 (Parent-Child Splitting)

책은 자식 벡터를 Chroma에, 부모 원문은 `InMemoryStore`(RAM)에 저장합니다.
이 노트북은 **부모 문서도 InMemoryStore 대신 Chroma 컬렉션에 key-value 형태로 저장**하여,
프로그램을 재시작해도 부모/자식 문서가 모두 영속적으로 남도록 구성합니다.

## 0. 환경 설정

In [ ]:
import chromadb

# --- LM Studio --- 
LMSTUDIO_BASE_URL = "http://...:.../v1"         # /v1 까지 포함
LMSTUDIO_API_KEY  = "lm-studio"                              # LM Studio는 키 검증 안 함 (더미)

EMBED_MODEL = "text-embedding-qwen3-embedding-8b"

# --- Chroma 서버 ---
CHROMA_HOST = "chromadb"
CHROMA_PORT = 8000

# --- 자식/부모 컬렉션 이름 ---
CHILD_COLLECTION  = "practice4_parent_child_children"   # 자식 청크(임베딩 검색용)
PARENT_COLLECTION = "practice4_parent_child_parents"     # 부모 원문(키-값 조회용)

# --- 입력 파일 (노트북 기준 상대경로) ---
DATA_PATH = "How_to_invest_money.txt"

chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
print("Chroma 서버 연결:", chroma_client.heartbeat())

In [ ]:
from langchain_openai import OpenAIEmbeddings

# 임베딩 모델
# check_embedding_ctx_length=False : tiktoken 대신 원문 문자열을 전송 → LM Studio 호환
# chunk_size=8 : 로컬 LM Studio 서버가 한 번에 받는 텍스트 수가 많으면 연결이 끊기므로 작게 배치
embeddings = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    check_embedding_ctx_length=False,
    chunk_size=8,
)

print("EMBED:", len(embeddings.embed_query("연결 테스트")), "차원")

## 1. 문서 로드

책은 구글 드라이브 경로에서 `How_to_invest_money.txt`를 읽지만, 로컬 환경에서는 노트북 기준 상대경로(`practice4/`)에서 읽습니다.

In [ ]:
from langchain_community.document_loaders import TextLoader

# 문서 로더 설정
# encoding="utf-8-sig" : 원본 파일 선두의 BOM(﻿) 제거
loaders = [
    TextLoader(DATA_PATH, encoding="utf-8-sig"),
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

print("로드된 문서 수:", len(docs))
print("전체 글자 수:", len(docs[0].page_content))

## 2. 부모-자식 분할 설정

재귀적 문자 텍스트 분할 방식을 활용하여 부모 문서와 자식 문서를 생성합니다. 부모 문서의 크기는 1000으로, 자식 문서의 크기는 200으로 설정함으로써 부모-자식 계층 구조를 만듭니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 부모 문서 생성을 위한 텍스트 분할기
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
# 자식 문서 생성을 위한 텍스트 분할기 (부모보다 작은 크기로 설정)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

## 3. 저장소 구성 (자식 벡터스토어 + 부모 문서 저장소, 전부 Chroma)

책은 자식 문서 인덱싱용 벡터 저장소(Chroma)와 부모 문서 저장을 위한 `InMemoryStore`(RAM)를 함께 사용합니다.
이 노트북에서는 RAM을 쓰지 않고 **부모 문서도 Chroma 컬렉션에 key-value 형태로 저장**합니다.

- `vectorstore`: 자식 청크를 임베딩하여 저장하는 Chroma 컬렉션 (`{CHILD_COLLECTION}`) — 의미 기반 검색용
- `parent_store`: 부모 원문을 id로 저장/조회하는 Chroma 컬렉션 (`{PARENT_COLLECTION}`) — `BaseStore` 인터페이스 구현

`parent_store`는 유사도 검색을 하지 않고 id로만 조회하므로, 임베딩 API 호출 비용을 없애기 위해 더미(0) 벡터를 사용합니다.

In [ ]:
from typing import Iterator, List, Optional, Sequence, Tuple
from langchain_core.stores import BaseStore
from langchain_core.documents import Document


class ChromaParentStore(BaseStore[str, Document]):
    """부모 문서를 Chroma 컬렉션에 key-value 형태로 저장하는 문서 저장소.
    id 기반 get/set/delete/iter 만 사용하므로 임베딩은 더미(0) 벡터를 넣어
    임베딩 API 호출 없이 저장한다."""

    def __init__(self, client: chromadb.HttpClient, collection_name: str, dim: int = 8):
        self._collection = client.get_or_create_collection(collection_name)
        self._dim = dim

    def mget(self, keys: Sequence[str]) -> List[Optional[Document]]:
        if not keys:
            return []
        res = self._collection.get(ids=list(keys), include=["documents", "metadatas"])
        found = {i: (d, m) for i, d, m in zip(res["ids"], res["documents"], res["metadatas"])}
        return [
            Document(page_content=found[k][0], metadata=found[k][1] or {}) if k in found else None
            for k in keys
        ]

    def mset(self, key_value_pairs: Sequence[Tuple[str, Document]]) -> None:
        if not key_value_pairs:
            return
        ids = [k for k, _ in key_value_pairs]
        texts = [v.page_content for _, v in key_value_pairs]
        metas = [v.metadata or {} for _, v in key_value_pairs]
        dummy_embeddings = [[0.0] * self._dim for _ in ids]
        self._collection.upsert(ids=ids, documents=texts, metadatas=metas, embeddings=dummy_embeddings)

    def mdelete(self, keys: Sequence[str]) -> None:
        if keys:
            self._collection.delete(ids=list(keys))

    def yield_keys(self, *, prefix: Optional[str] = None) -> Iterator[str]:
        res = self._collection.get(include=[])
        for k in res["ids"]:
            if prefix is None or k.startswith(prefix):
                yield k

In [ ]:
from langchain_chroma import Chroma

# 재실행 안전: 이전 실행에서 남은 컬렉션 제거 후 새로 생성
for name in (CHILD_COLLECTION, PARENT_COLLECTION):
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass

# 자식 문서 인덱싱을 위한 벡터 저장소
vectorstore = Chroma(
    client=chroma_client,
    collection_name=CHILD_COLLECTION,
    embedding_function=embeddings,
)
# 부모 문서 저장을 위한 저장소 (Chroma 기반)
parent_store = ChromaParentStore(chroma_client, PARENT_COLLECTION)

print("vectorstore, parent_store 준비 완료")

## 4. ParentDocumentRetriever 생성 및 문서 추가

자식 문서 벡터 저장소(`vectorstore`), 부모 문서 저장소(`parent_store`), 자식 문서 텍스트 분할기(`child_splitter`), 부모 문서 텍스트 분할기(`parent_splitter`)를 인자로 받는 `ParentDocumentRetriever` 인스턴스를 생성합니다. 이를 통해 부모-자식 구조의 계층적 문서 검색을 수행할 수 있습니다.

`langchain` 1.x 에서 `ParentDocumentRetriever` 의 위치가 `langchain_classic` 으로 이동했으므로 폴백 임포트를 사용합니다.

In [ ]:
# ParentDocumentRetriever: langchain 1.x 에서 위치 이동 → 폴백 처리
try:
    from langchain_classic.retrievers import ParentDocumentRetriever
except ImportError:
    from langchain.retrievers import ParentDocumentRetriever

# ParentDocumentRetriever 설정
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=parent_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 문서 추가
retriever.add_documents(docs)

# 부모 문서 수 확인
print(f"Number of parent documents: {len(list(parent_store.yield_keys()))}")

## 5. 검색 확인

"What are the types of investments?"라고 질의해 보겠습니다. 이 질문이 `ParentDocumentRetriever`를 거치면 관련된 부모 문서가 검색됩니다. 여기서는 첫 번째 연관 문서만 출력해서 확인해 보겠습니다.

In [ ]:
# 질문 정의
query = "What are the types of investments?"

# 연관 문서 수집 (langchain 1.x: get_relevant_documents 대신 invoke 사용)
retrieved_docs = retriever.invoke(query)

# 첫 번째 연관 문서 출력
print(f"Parent Document: {retrieved_docs[0].page_content}")

이제, 부모-자식 분할의 작동 방식을 이해하기 위해 벡터 저장소에서 직접 자식 문서를 검색해서 첫 번째 자식 문서를 출력해 보겠습니다.

In [ ]:
# 자식 문서 검색
query = "What are the types of investments?"
sub_docs = vectorstore.similarity_search(query)
print(f"Child Document: {sub_docs[0].page_content}")